In [49]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from collections import Counter
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import os
import mlflow
import mlflow.pytorch

os.chdir(r"C:\Users\hempe\Studium\Masterthesis\Repository\Masterthesis")

In [ ]:
#load data from CSV with unicode encoding
df = pd.read_csv("data/processed/train_data.csv", sep=',', encoding='utf-8')

C:\Users\hempe\AppData\Local\Temp\ipykernel_27344\2260802705.py:2: DtypeWarning: Columns (0: Kd (nM), 1: EC50 (nM), 2: koff (s-1), 3: UniProt (TrEMBL) Submitted Name of Target Chain 2, 4: UniProt (TrEMBL) Entry Name of Target Chain 2, 5: UniProt (TrEMBL) Primary ID of Target Chain 2, 6: UniProt (TrEMBL) Secondary ID(s) of Target Chain 2, 7: BindingDB Target Chain Sequence 3, 8: PDB ID(s) of Target Chain 3, 9: UniProt (SwissProt) Recommended Name of Target Chain 3, 10: UniProt (SwissProt) Entry Name of Target Chain 3, 11: UniProt (SwissProt) Primary ID of Target Chain 3, 12: UniProt (SwissProt) Secondary ID(s) of Target Chain 3, 13: UniProt (TrEMBL) Submitted Name of Target Chain 3, 14: UniProt (TrEMBL) Entry Name of Target Chain 3, 15: UniProt (TrEMBL) Primary ID of Target Chain 3, 16: UniProt (TrEMBL) Secondary ID(s) of Target Chain 3, 17: BindingDB Target Chain Sequence 4, 18: PDB ID(s) of Target Chain 4, 19: UniProt (SwissProt) Recommended Name of Target Chain 4, 20: UniProt (SwissP

In [17]:
df = df[['Ligand SMILES', 'BindingDB Target Chain Sequence 1', 'IC50 (nM)']].dropna()
df['IC50 (nM)'] = pd.to_numeric(df['IC50 (nM)'], errors='coerce')


In [18]:
# gib me an overview of all missing values in the dataset
print(df.isnull().sum())

Ligand SMILES                           0
BindingDB Target Chain Sequence 1       0
IC50 (nM)                            6938
dtype: int64


In [19]:
df['IC50 (nM)'] = pd.to_numeric(df['IC50 (nM)'], errors='coerce')

In [20]:
# gib me an overview of all missing values in the dataset
print(df.isnull().sum())

Ligand SMILES                           0
BindingDB Target Chain Sequence 1       0
IC50 (nM)                            6938
dtype: int64


In [21]:
df = df.dropna()

In [22]:
# gib me an overview of all missing values in the dataset
print(df.isnull().sum())

Ligand SMILES                        0
BindingDB Target Chain Sequence 1    0
IC50 (nM)                            0
dtype: int64


In [23]:
# show me the datatypes of the columns
print(df.dtypes)

Ligand SMILES                            str
BindingDB Target Chain Sequence 1        str
IC50 (nM)                            float64
dtype: object


In [24]:
# filter all data where IC50 (nM) is not null
df = df[df['IC50 (nM)'].notnull()]
#filter all data where BindingDB Target Chain Sequence 1 is not null
df = df[df['BindingDB Target Chain Sequence 1'].notnull()]
#filter all data where Ligand SMILES is not null
df = df[df['Ligand SMILES'].notnull()]
#filter all data where IC50 (nM) is not numeric
df = df[pd.to_numeric(df['IC50 (nM)'], errors='coerce').notnull()]
#transform column IC50 (nM) to numeric
df["IC50 (nM)"] = pd.to_numeric(df["IC50 (nM)"], errors="coerce")
#filter all rows were IC50 is < 0
df = df[df["IC50 (nM)"] > 0]
#Remove outlier >1e7
df = df[df["IC50 (nM)"] <= 1e7]


In [25]:
#Select relevant feaures
df=df[['Ligand SMILES', 'BindingDB Target Chain Sequence 1', 'IC50 (nM)']]
df.head()

,Ligand SMILES,BindingDB Target Chain Sequence 1,IC50 (nM)
1,CCc1cn2CCS(=O)(=O)Oc3cc(cc1c23)C(=O)N[C@@H](Cc...,MGALARALLLPLLAQWLLRAAPELAPAPFTLPLRVAAATNRVVAPT...,450.0
2,Clc1cccc(Nc2ncnc3n[nH]c(NCc4ccccc4)c23)c1,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,7.0
3,CC1CCCCCCCC(=O)Cc2c(Cl)c(O)cc(O)c2C(=O)O1,MASETFEFQAEITQLMSLIINTVYSNKEIFLRELISNASDALDKIR...,1290.0
4,COc1cc(ccc1N)-c1ccc2c(n[nH]c2c1)-c1nc2c(cccc2[...,MPSRTGPKMEGSGGRVRLKAHYGGDIFITSVDAATTFEELCEEVRD...,51.3
7,Cc1ccc(cc1)N1CCc2cc(O)ccc2C1(C)c1ccc(OCCN2CCCC...,MDIKNSPSSLNSPSSYNCSQSILPLEHGSIYIPSSYVDSHHEYPAM...,354.0


In [26]:
#first row of the column ligand SMILES
smiles_example = df['Ligand SMILES'].iloc[0]

smiles_example


'CCc1cn2CCS(=O)(=O)Oc3cc(cc1c23)C(=O)N[C@@H](Cc1ccccc1)[C@H](O)CNC1CCOCC1 |r|'

In [27]:
#first row of the column ligand SMILES
protein_example = df['BindingDB Target Chain Sequence 1'].iloc[0]

protein_example

'MGALARALLLPLLAQWLLRAAPELAPAPFTLPLRVAAATNRVVAPTPGPGTPAERHADGLALALEPALASPAGAANFLAMVDNLQGDSGRGYYLEMLIGTPPQKLQILVDTGSSNFAVAGTPHSYIDTYFDTERSSTYRSKGFDVTVKYTQGSWTGFVGEDLVTIPKGFNTSFLVNIATIFESENFFLPGIKWNGILGLAYATLAKPSSSLETFFDSLVTQANIPNVFSMQMCGAGLPVAGSGTNGGSLVLGGIEPSLYKGDIWYTPIKEEWYYQIEILKLEIGGQSLNLDCREYNADKAIVDSGTTLLRLPQKVFDAVVEAVARASLIPEFSDGFWTGSQLACWTNSETPWSYFPKISIYLRDENSSRSFRITILPQLYIQPMMGAGLNYECYRFGISPSTNALVIGATVMEGFYVIFDRAQKRVGFAASPCAEIAGAAVSEISGPFSTEDVASNCVPAQSLSEPILWIVSYALMSVCGAILLVLIVLLLLPFRCQRRPRDPEVVNDESSLVRHRWK'

In [28]:
# SMILES-Zeichen-Wörterbuch aus der Arbeit
CHARISOSMISET = {
    "#": 29, "%": 30, ")": 31, "(": 1, "+": 32, "-": 33, "/": 34, ".": 2,
    "1": 35, "0": 3, "3": 36, "2": 4, "5": 37, "4": 5, "7": 38, "6": 6,
    "9": 39, "8": 7, "=": 40, "A": 41, "@": 8, "C": 42, "B": 9, "E": 43,
    "D": 10, "G": 44, "F": 11, "I": 45, "H": 12, "K": 46, "M": 47, "L": 13,
    "O": 48, "N": 14, "P": 15, "S": 49, "R": 16, "U": 50, "T": 17, "W": 51,
    "V": 18, "Y": 52, "[": 53, "Z": 19, "]": 54, "\\": 20, "a": 55, "c": 56,
    "b": 21, "e": 57, "d": 22, "g": 58, "f": 23, "i": 59, "h": 24, "m": 60,
    "l": 25, "o": 61, "n": 26, "s": 62, "r": 27, "u": 63, "t": 28, "y": 64
}

# Protein-Aminosäuren-Wörterbuch
CHARPROTSET = {
    "A": 1, "C": 2, "B": 3, "E": 4, "D": 5, "G": 6,
    "F": 7, "I": 8, "H": 9, "K": 10, "M": 11, "L": 12,
    "O": 13, "N": 14, "Q": 15, "P": 16, "S": 17, "R": 18,
    "U": 19, "T": 20, "W": 21, "V": 22, "Y": 23, "X": 24, "Z": 25
}

In [29]:
protein_integer_sequence = []

for char in protein_example:
    protein_integer_sequence.append(
        CHARPROTSET.get(char, 0)
    )
protein_integer_sequence

[11,
 6,
 1,
 12,
 1,
 18,
 1,
 12,
 12,
 12,
 16,
 12,
 12,
 1,
 15,
 21,
 12,
 12,
 18,
 1,
 1,
 16,
 4,
 12,
 1,
 16,
 1,
 16,
 7,
 20,
 12,
 16,
 12,
 18,
 22,
 1,
 1,
 1,
 20,
 14,
 18,
 22,
 22,
 1,
 16,
 20,
 16,
 6,
 16,
 6,
 20,
 16,
 1,
 4,
 18,
 9,
 1,
 5,
 6,
 12,
 1,
 12,
 1,
 12,
 4,
 16,
 1,
 12,
 1,
 17,
 16,
 1,
 6,
 1,
 1,
 14,
 7,
 12,
 1,
 11,
 22,
 5,
 14,
 12,
 15,
 6,
 5,
 17,
 6,
 18,
 6,
 23,
 23,
 12,
 4,
 11,
 12,
 8,
 6,
 20,
 16,
 16,
 15,
 10,
 12,
 15,
 8,
 12,
 22,
 5,
 20,
 6,
 17,
 17,
 14,
 7,
 1,
 22,
 1,
 6,
 20,
 16,
 9,
 17,
 23,
 8,
 5,
 20,
 23,
 7,
 5,
 20,
 4,
 18,
 17,
 17,
 20,
 23,
 18,
 17,
 10,
 6,
 7,
 5,
 22,
 20,
 22,
 10,
 23,
 20,
 15,
 6,
 17,
 21,
 20,
 6,
 7,
 22,
 6,
 4,
 5,
 12,
 22,
 20,
 8,
 16,
 10,
 6,
 7,
 14,
 20,
 17,
 7,
 12,
 22,
 14,
 8,
 1,
 20,
 8,
 7,
 4,
 17,
 4,
 14,
 7,
 7,
 12,
 16,
 6,
 8,
 10,
 21,
 14,
 6,
 8,
 12,
 6,
 12,
 1,
 23,
 1,
 20,
 12,
 1,
 10,
 16,
 17,
 17,
 17,
 12,
 4,
 20,
 7,
 7,
 5,
 17,
 12

In [30]:
smiles_integer_sequence = []

for char in smiles_example:
    smiles_integer_sequence.append(
        CHARISOSMISET.get(char, 0)
    )
smiles_integer_sequence

[42,
 42,
 56,
 35,
 56,
 26,
 4,
 42,
 42,
 49,
 1,
 40,
 48,
 31,
 1,
 40,
 48,
 31,
 48,
 56,
 36,
 56,
 56,
 1,
 56,
 56,
 35,
 56,
 4,
 36,
 31,
 42,
 1,
 40,
 48,
 31,
 14,
 53,
 42,
 8,
 8,
 12,
 54,
 1,
 42,
 56,
 35,
 56,
 56,
 56,
 56,
 56,
 35,
 31,
 53,
 42,
 8,
 12,
 54,
 1,
 48,
 31,
 42,
 14,
 42,
 35,
 42,
 42,
 48,
 42,
 42,
 35,
 0,
 0,
 27,
 0]

In [31]:
def label_smiles(smiles, max_len=100):
    x = np.zeros(max_len, dtype=np.int64)
    
    for i, ch in enumerate(smiles[:max_len]):
        x[i] = CHARISOSMISET.get(ch, 0)  # unbekannte Zeichen -> 0
        
    return x


def label_protein(sequence, max_len=1200):
    x = np.zeros(max_len, dtype=np.int64)
    
    for i, ch in enumerate(sequence[:max_len]):
        x[i] = CHARPROTSET.get(ch, 0)  # unbekannte Aminosäuren -> 0
        
    return x

In [32]:
df_model = df[
    ['Ligand SMILES', 'BindingDB Target Chain Sequence 1', 'IC50 (nM)']
].dropna().copy()

df_model['smiles_encoded'] = df_model['Ligand SMILES'].apply(label_smiles)
df_model['protein_encoded'] = df_model['BindingDB Target Chain Sequence 1'].apply(label_protein)

In [33]:
df_model['pIC50'] = -np.log10(df_model['IC50 (nM)'].astype(float) * 1e-9)

In [34]:
X_smiles = np.stack(df_model['smiles_encoded'].values)
X_protein = np.stack(df_model['protein_encoded'].values)
y = df_model['pIC50'].values.astype(np.float32)

print(X_smiles.shape)
print(X_protein.shape)
print(y.shape)

(43966, 100)
(43966, 1200)
(43966,)


In [35]:
X_smiles

array([[42, 42, 56, ...,  0,  0,  0],
       [42, 25, 56, ...,  0,  0,  0],
       [42, 42, 35, ...,  0,  0,  0],
       ...,
       [42, 42, 14, ...,  0,  0,  0],
       [42, 42,  1, ...,  0,  0,  0],
       [42, 25, 56, ...,  0,  0,  0]], dtype=int64)

In [36]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split

In [37]:
X_smiles_tensor = torch.tensor(X_smiles, dtype=torch.long)
X_protein_tensor = torch.tensor(X_protein, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)

In [41]:
X_smiles_tensor

tensor([[42, 42, 56,  ...,  0,  0,  0],
        [42, 25, 56,  ...,  0,  0,  0],
        [42, 42, 35,  ...,  0,  0,  0],
        ...,
        [42, 42, 14,  ...,  0,  0,  0],
        [42, 42,  1,  ...,  0,  0,  0],
        [42, 25, 56,  ...,  0,  0,  0]])

In [38]:
dataset = TensorDataset(X_smiles_tensor, X_protein_tensor, y_tensor)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [46]:
torch.set_printoptions(threshold=1000000)
dataset[0]

(tensor([42, 42, 56, 35, 56, 26,  4, 42, 42, 49,  1, 40, 48, 31,  1, 40, 48, 31,
         48, 56, 36, 56, 56,  1, 56, 56, 35, 56,  4, 36, 31, 42,  1, 40, 48, 31,
         14, 53, 42,  8,  8, 12, 54,  1, 42, 56, 35, 56, 56, 56, 56, 56, 35, 31,
         53, 42,  8, 12, 54,  1, 48, 31, 42, 14, 42, 35, 42, 42, 48, 42, 42, 35,
          0,  0, 27,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0]),
 tensor([11,  6,  1, 12,  1, 18,  1, 12, 12, 12, 16, 12, 12,  1, 15, 21, 12, 12,
         18,  1,  1, 16,  4, 12,  1, 16,  1, 16,  7, 20, 12, 16, 12, 18, 22,  1,
          1,  1, 20, 14, 18, 22, 22,  1, 16, 20, 16,  6, 16,  6, 20, 16,  1,  4,
         18,  9,  1,  5,  6, 12,  1, 12,  1, 12,  4, 16,  1, 12,  1, 17, 16,  1,
          6,  1,  1, 14,  7, 12,  1, 11, 22,  5, 14, 12, 15,  6,  5, 17,  6, 18,
          6, 23, 23, 12,  4, 11, 12,  8,  6, 20, 16, 16, 15, 10, 12, 15,  8, 12,
         22,  5, 20,  6, 17, 17, 14,  7,  1, 22,  1,  6, 2

In [39]:
class SimpleAttentionDTA(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.smiles_embedding = nn.Embedding(65, 128, padding_idx=0)
        self.protein_embedding = nn.Embedding(26, 128, padding_idx=0)
        
        self.smiles_cnn = nn.Conv1d(128, 96, kernel_size=8)
        self.protein_cnn = nn.Conv1d(128, 96, kernel_size=12)
        
        self.pool = nn.AdaptiveMaxPool1d(1)
        
        self.fc = nn.Sequential(
            nn.Linear(192, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
        
    def forward(self, smiles, protein):
        smiles = self.smiles_embedding(smiles)
        protein = self.protein_embedding(protein)
        
        smiles = smiles.permute(0, 2, 1)
        protein = protein.permute(0, 2, 1)
        
        smiles = torch.relu(self.smiles_cnn(smiles))
        protein = torch.relu(self.protein_cnn(protein))
        
        smiles = self.pool(smiles).squeeze(-1)
        protein = self.pool(protein).squeeze(-1)
        
        combined = torch.cat([smiles, protein], dim=1)
        output = self.fc(combined)
        
        return output

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleAttentionDTA().to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

epochs = 20

for epoch in range(epochs):
    model.train()
    train_losses = []
    
    for smiles_batch, protein_batch, y_batch in train_loader:
        smiles_batch = smiles_batch.to(device)
        protein_batch = protein_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        
        predictions = model(smiles_batch, protein_batch)
        loss = loss_fn(predictions, y_batch)
        
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
    
    print(f"Epoch {epoch+1}, Train MSE: {sum(train_losses)/len(train_losses):.4f}")

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

model.eval()

all_preds = []
all_true = []

with torch.no_grad():
    for smiles_batch, protein_batch, y_batch in test_loader:
        smiles_batch = smiles_batch.to(device)
        protein_batch = protein_batch.to(device)
        
        preds = model(smiles_batch, protein_batch)
        
        all_preds.extend(preds.cpu().numpy().flatten())
        all_true.extend(y_batch.numpy().flatten())

mse = mean_squared_error(all_true, all_preds)
mae = mean_absolute_error(all_true, all_preds)
r2 = r2_score(all_true, all_preds)

print("Test MSE:", mse)
print("Test MAE:", mae)
print("Test R2:", r2)

Test MSE: 0.7260551541284613
Test MAE: 0.658765811743399
Test R2: 0.6469767055322626


In [48]:
import mlflow; print(mlflow.__version__)

3.12.0
